# Zn ExpertXAS: evaluate 310–500–750

Evaluate the saved **310–500–750** Zn FEFF head on its original encoder features. This notebook does not train, sweep, or select a model. All plots and metrics use this one head.

The first cell checks for the local checkpoint, encoder, exported features, canonical dataset, and provenance. Missing files stop execution. This exploratory dataset uses unscaled targets and is not directly comparable with the paper.


In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import sys
from unittest.mock import patch

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "omnixas").is_dir())
sys.path.insert(0, str(REPO_ROOT))
DATA_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO_ROOT.parent / "OmniXAS_data")).expanduser().resolve()
DATASET_ROOT = DATA_ROOT / "figshare_141_splits_unscaled_zn9"
RAW_ROOT = DATA_ROOT / "figshare"
RUN_ROOT = REPO_ROOT / "output" / "training" / "zn_expert_sweep"
ENCODER_RUN = RUN_ROOT / "zn9_universal_encoder"
FEATURES_DIR = ENCODER_RUN / "features"
HEAD_DIR = RUN_ROOT / "expert_sweep_tens_200" / "heads" / "310x500x750"
OUTPUT_DIR = RUN_ROOT / "evaluation_310x500x750"
DIMS = [310, 500, 750]
SPLITS = ("train", "val", "test")
ENERGY_GRID = np.linspace(9659.0, 9694.0, 141)
SIGMA_EV = 0.5
BATCH_SIZE = 4096

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

required = [HEAD_DIR / "best.pt", HEAD_DIR / "metrics.json", ENCODER_RUN / "best_encoder.ckpt",
            ENCODER_RUN / "RUN_COMPLETE.json", ENCODER_RUN / "notebook_dataset_sha256.txt",
            DATASET_ROOT / "dataset.json"]
for split in SPLITS:
    required += [FEATURES_DIR / f"Zn_FEFF_{split}_{suffix}.txt" for suffix in ("X", "y", "ids")]
    required += [DATASET_ROOT / "material_id_and_site" / f"Zn_FEFF_{split}.txt",
                 DATASET_ROOT / "ml_data" / f"Zn_FEFF_{split}_y.txt"]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError("Missing required local artifacts:\n" + "\n".join(missing))
print("All required local artifacts found.")


## Verify the checkpoint and row order

The head must match its recorded checkpoint, encoder, dataset, and training and validation feature hashes. Test IDs and targets must match the canonical dataset. These checks do not independently prove that test feature rows were generated in the correct order.


In [ ]:
import torch
from omnixas.model.xasblock import XASBlock

record = json.loads((HEAD_DIR / "metrics.json").read_text(encoding="utf-8"))
provenance = record.get("provenance", {})
if record.get("hidden_dims") != "310-500-750" or record.get("checkpoint_sha256") != sha256(HEAD_DIR / "best.pt"):
    raise ValueError("The head checkpoint or its architecture does not match its recorded provenance.")
if provenance.get("encoder_sha256") != sha256(ENCODER_RUN / "best_encoder.ckpt"):
    raise ValueError("The saved head and encoder do not match.")
dataset_hash = sha256(DATASET_ROOT / "dataset.json")
if provenance.get("dataset_sha256") != dataset_hash or (ENCODER_RUN / "notebook_dataset_sha256.txt").read_text().strip() != dataset_hash:
    raise ValueError("The head, encoder, and canonical dataset provenance do not match.")
if json.loads((ENCODER_RUN / "RUN_COMPLETE.json").read_text()).get("stage") != "encoder_only":
    raise ValueError("The encoder run is not marked complete.")
for split in ("train", "val"):
    if provenance.get("feature_sha256", {}).get(split) != sha256(FEATURES_DIR / f"Zn_FEFF_{split}_X.txt"):
        raise ValueError(f"The {split} features do not match the head's recorded features.")

parts = {}
for split in SPLITS:
    prefix = f"Zn_FEFF_{split}"
    X = np.loadtxt(FEATURES_DIR / f"{prefix}_X.txt", dtype=np.float32, ndmin=2)
    y = np.loadtxt(FEATURES_DIR / f"{prefix}_y.txt", dtype=np.float32, ndmin=2)
    ids = (FEATURES_DIR / f"{prefix}_ids.txt").read_text(encoding="utf-8").splitlines()
    canonical_ids = (DATASET_ROOT / "material_id_and_site" / f"{prefix}.txt").read_text(encoding="utf-8-sig").splitlines()
    canonical_y = np.loadtxt(DATASET_ROOT / "ml_data" / f"{prefix}_y.txt", dtype=np.float32, ndmin=2)
    if X.shape != (len(ids), 64) or y.shape != (len(ids), 141) or not np.isfinite(X).all() or not np.isfinite(y).all():
        raise ValueError(f"Invalid feature or spectrum dimensions for {split}.")
    if ids != canonical_ids or not np.array_equal(y, canonical_y) or len(ids) != len(set(ids)):
        raise ValueError(f"ID or target mismatch for {split}.")
    parts[split] = (ids, X, y)
material_sets = [{value.rsplit("_", 1)[0] for value in parts[split][0]} for split in SPLITS]
if any(material_sets[i] & material_sets[j] for i in range(3) for j in range(i + 1, 3)):
    raise ValueError("A Zn material appears in multiple splits.")

state = torch.load(HEAD_DIR / "best.pt", map_location="cpu", weights_only=False)
if state.get("hidden_dims") != DIMS or float(state.get("dropout", -1)) != 0.05:
    raise ValueError("Checkpoint is not the expected 310–500–750 head.")
with patch.object(XASBlock, "DROPOUT", float(state["dropout"])):
    head = XASBlock(64, DIMS, 141)
head.load_state_dict(state["state_dict"], strict=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
head.to(device).eval()
print(f"Verified 310–500–750 head: {sum(p.numel() for p in head.parameters()):,} trainable parameters; device: {device}.")


## Evaluate Zn test spectra

The baseline is the mean Zn training spectrum. Eta is the median per-spectrum MSE of that baseline divided by the median per-spectrum MSE of the head. Higher eta is better.


In [ ]:
test_ids, test_X, test_y = parts["test"]
train_mean = parts["train"][2].mean(axis=0)
with torch.inference_mode():
    predictions = torch.cat([head(batch.to(device)).cpu() for batch in torch.from_numpy(test_X).split(BATCH_SIZE)]).numpy()
if predictions.shape != test_y.shape or not np.isfinite(predictions).all():
    raise ValueError("Invalid test predictions.")
errors = ((predictions - test_y) ** 2).mean(axis=1)
baseline_errors = ((test_y - train_mean) ** 2).mean(axis=1)
median_mse = float(np.median(errors))
baseline_median = float(np.median(baseline_errors))
eta = baseline_median / median_mse if median_mse else (float("inf") if baseline_median else float("nan"))
print(f"310–500–750 | test spectra: {len(test_ids)} | MSE: {errors.mean():.6g} | median MSE: {median_mse:.6g} | eta: {eta:.3f}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
np.savez_compressed(OUTPUT_DIR / "test_predictions.npz", predictions=predictions, targets=test_y, ids=np.asarray(test_ids))
(OUTPUT_DIR / "test_metrics.json").write_text(json.dumps({"hidden_dims": DIMS, "head_sha256": sha256(HEAD_DIR / "best.pt"),
    "encoder_sha256": provenance["encoder_sha256"], "dataset_sha256": dataset_hash,
    "test_count": len(test_ids), "mse": float(errors.mean()), "median_mse": median_mse,
    "baseline_median_mse": baseline_median, "eta": eta}, indent=2), encoding="utf-8")


## Test error and spectrum detail

Each dot is one **test** spectrum. Farther right means a larger prediction error. Higher up means the target spectrum has more sharp or rapidly changing detail. The right panel zooms in on most dots. The red dot marks the largest error.

The detail ratio compares each target with a Gaussian-smoothed version (sigma 0.5 eV). Sharp detail can be a real spectral feature, not noise. Both correlations use all test spectra; correlation does not prove a cause.


In [ ]:
from scipy.stats import pearsonr, spearmanr

def noise_ratio(y):
    scale = float(np.std(y))
    if scale == 0:
        return 0.0
    sigma_points = SIGMA_EV / (ENERGY_GRID[1] - ENERGY_GRID[0])
    half = max(1, int(3 * sigma_points))
    x = np.arange(-half, half + 1)
    kernel = np.exp(-0.5 * (x / sigma_points) ** 2)
    kernel /= kernel.sum()
    smooth = np.convolve(np.pad(y, half, mode="edge"), kernel, mode="valid")
    return float(np.std(y - smooth) / scale)

ratios = np.array([noise_ratio(y) for y in test_y])
pearson = float(pearsonr(errors, ratios).statistic) if np.ptp(errors) and np.ptp(ratios) else float("nan")
spearman = float(spearmanr(errors, ratios).statistic) if np.ptp(errors) and np.ptp(ratios) else float("nan")
chart = pd.DataFrame({"id": test_ids, "mse": errors, "noise_ratio": ratios})
chart[["material_id", "site"]] = chart["id"].str.rsplit("_", n=1, expand=True)
chart.to_csv(OUTPUT_DIR / "test_noise_ratio.csv", index=False)
worst = int(np.argmax(errors))
xmax = max(float(np.percentile(errors, 99)), np.finfo(float).eps)
ymax = max(float(np.percentile(ratios, 99)), np.finfo(float).eps)
visible = (errors <= xmax) & (ratios <= ymax)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax in axes:
    ax.scatter(errors, ratios, s=18, alpha=0.6)
    ax.set_xlabel("Per-spectrum MSE (test)")
    ax.set_ylabel("High-frequency ratio (Gaussian sigma = 0.5 eV)")
    ax.grid(alpha=0.3)
axes[0].scatter(errors[worst], ratios[worst], s=55, color="tab:red")
axes[0].annotate(f"Zn {chart.material_id.iloc[worst]} site {chart.site.iloc[worst]}",
                 (errors[worst], ratios[worst]), xytext=(-8, -12), textcoords="offset points", ha="right", fontsize=8)
axes[0].set_title(f"All {len(errors)} spectra | Pearson r = {pearson:.2f} | Spearman rho = {spearman:.2f}")
axes[1].set_xlim(0, xmax)
axes[1].set_ylim(0, ymax)
axes[1].set_title(f"99th-percentile zoom | {int(visible.sum())} of {len(errors)} shown")
fig.suptitle("Zn test: 310–500–750")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "mse_vs_noise.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)
display(chart.sort_values("mse", ascending=False).head(5))


## Test spectra by MSE percentile

The first grid shows spectra near nine MSE percentiles. The second shows nine distinct materials near the 95th percentile. Labels include the element, material ID, absorbing site, and MSE.


In [ ]:
percentiles = [5, 10, 20, 30, 50, 70, 80, 90, 95]
selected = []
for q in percentiles:
    order = np.argsort(abs(errors - np.percentile(errors, q)), kind="stable")
    i = next((int(index) for index in order if int(index) not in selected), None)
    if i is None:
        raise ValueError("At least nine test spectra are needed.")
    selected.append(i)

def spectrum_grid(indices, title, filename, percentile_labels=None):
    fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True)
    for position, (ax, i) in enumerate(zip(axes.flat, indices)):
        ax.plot(ENERGY_GRID, test_y[i], label="Target")
        ax.plot(ENERGY_GRID, predictions[i], label="Prediction")
        mid, site = test_ids[i].rsplit("_", 1)
        percentile = f" | {percentile_labels[position]}th percentile" if percentile_labels is not None else ""
        ax.set_title(f"Zn {mid} site {site}{percentile}\nMSE {errors[i]:.4g}", fontsize=9)
        ax.set_xlabel("Energy (eV)")
        ax.set_ylabel("Intensity (unscaled)")
        ax.grid(alpha=0.2)
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / filename, dpi=180)
    display(fig)
    plt.close(fig)

spectrum_grid(selected, "Zn 310–500–750: test MSE percentiles", "test_percentiles.png", percentiles)
q95 = float(np.percentile(errors, 95))
near95, seen = [], set()
for i in np.argsort(abs(errors - q95), kind="stable"):
    mid = test_ids[i].rsplit("_", 1)[0]
    if mid not in seen:
        seen.add(mid)
        near95.append(int(i))
    if len(near95) == 9:
        break
if len(near95) < 9:
    raise ValueError("At least nine distinct test materials are needed.")
spectrum_grid(near95, f"Zn 310–500–750: distinct materials near 95th-percentile MSE ({q95:.4g})", "test_near_p95.png")


## Iodine-containing Zn spectra

The main metrics and plots above use the **test** split. This iodine comparison also includes training spectra so you can compare their shapes. Each panel labels its split. **TEST** means the spectrum was held out of training; **TRAIN** means the model trained on it.

Each title shows baseline MSE divided by model MSE for that spectrum. This is not the paper's dataset-level eta. Source `structure.json` files must be available.


In [ ]:
iodine = []
for split in SPLITS:
    ids, X, y = parts[split]
    selected = []
    for i, value in enumerate(ids):
        mid, site = value.rsplit("_", 1)
        path = RAW_ROOT / "FEFF" / "Zn" / mid / "FEFF-XANES" / f"{int(site):03d}_Zn" / "structure.json"
        if not path.is_file():
            raise FileNotFoundError(f"Missing source structure: {path}")
        structure = json.loads(path.read_text(encoding="utf-8"))
        if any(species.get("element") == "I" for atom in structure["sites"] for species in atom["species"]):
            selected.append(i)
    if not selected:
        continue
    with torch.inference_mode():
        batches = torch.from_numpy(X[selected]).split(BATCH_SIZE)
        split_predictions = torch.cat([head(batch.to(device)).cpu() for batch in batches]).numpy()
    for i, prediction in zip(selected, split_predictions):
        target = y[i]
        model_mse = float(np.mean((target - prediction) ** 2))
        baseline_mse = float(np.mean((target - train_mean) ** 2))
        ratio = baseline_mse / model_mse if model_mse else (float("inf") if baseline_mse else float("nan"))
        iodine.append((ids[i], split, target, prediction, ratio))

iodine.sort(key=lambda item: {"test": 0, "val": 1, "train": 2}[item[1]])

if not iodine:
    print("No iodine-containing Zn spectra found.")
else:
    print("Iodine-containing Zn spectra by split:", pd.Series([row[1] for row in iodine]).value_counts().to_dict())
    fig, axes = plt.subplots(math.ceil(len(iodine) / 3), 3, figsize=(15, 3.8 * math.ceil(len(iodine) / 3)), squeeze=False, sharex=True)
    for ax, (value, split, target, prediction, ratio) in zip(axes.flat, iodine):
        ax.plot(ENERGY_GRID, target, label="Target")
        ax.plot(ENERGY_GRID, prediction, label="Prediction")
        mid, site = value.rsplit("_", 1)
        split_label = {"test": "TEST — held out", "train": "TRAIN — used for training", "val": "VALIDATION — held out"}[split]
        ax.set_title(f"Zn {mid} site {site} | {split_label}\nBaseline MSE / model MSE = {ratio:.2f}",
                     color="darkred" if split == "test" else "black", fontweight="bold" if split == "test" else "normal", fontsize=9)
        ax.set_xlabel("Energy (eV)")
        ax.set_ylabel("Intensity (unscaled)")
        ax.grid(alpha=0.2)
    for ax in axes.flat[len(iodine):]:
        ax.set_visible(False)
    axes.flat[0].legend()
    fig.suptitle("Iodine-containing Zn spectra: test and training comparison | 310–500–750")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "iodine_spectra.png", dpi=180, bbox_inches="tight")
    display(fig)
    plt.close(fig)
